# Singapore Career Intelligence: Skills Roadmap Analysis

This notebook is an expanded version of the job analytics project, incorporating advanced data cleaning, seniority classification, and a custom **Employability Score** calculation to help jobseekers prioritize high-value skills.

## 1. Business Case & Objectives

- Goal: Create an interactive career intelligence dashboard for the Singapore job market.

- Key Metrics: Industry-specific skill demand, average salaries, and seniority-based career pathways.

- Success Criteria: Automate the extraction of skills from "dirty" CSV data and rank them using a multi-factor Employability Score.

## 2. Setup and Database Connection
We use duckdb as an in-memory SQL engine to handle large CSV datasets efficiently without the overhead of a full database server.

In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Initialize in-memory connection
con = duckdb.connect(database=':memory:')

# File paths for raw data
jobs_csv = "temp_sgjobdata_cleaned-20k.csv"
skills_csv = "temp_job_skills_cleaned-10k.csv"

## 3. Data Enrichment and SQL Views
In this step, we use SQL to perform complex string manipulation. We clean the currency strings and classify job seniority using Regular Expressions (Regex).

In [3]:
# Create a cleaned view of the jobs data
con.execute(f"""
    CREATE OR REPLACE VIEW sg_jobs_raw AS 
    SELECT * FROM read_csv_auto('{jobs_csv}', all_varchar=True);
    
    CREATE OR REPLACE VIEW sg_jobs AS
    SELECT 
        categories,
        title,
        -- Clean average_salary by removing non-numeric characters and casting to DOUBLE
        TRY_CAST(regexp_replace(average_salary, '[^0-9.]', '', 'g') AS DOUBLE) as average_salary,
        -- Classify Seniority based on keywords in the Job Title
        CASE 
            WHEN lower(title) SIMILAR TO '%(intern|trainee|graduate|fresh|entry|junior)%' 
                 OR lower(positionLevels) LIKE '%fresh%' THEN 'Entry-level'
            WHEN lower(title) SIMILAR TO '%(senior|lead|principal|manager|head|director|architect)%' 
                 OR lower(positionLevels) SIMILAR TO '%(senior|manager)%' THEN 'Senior-level'
            ELSE 'Mid-level'
        END AS seniority
    FROM sg_jobs_raw
    WHERE title != 'title' 
      AND TRY_CAST(regexp_replace(average_salary, '[^0-9.]', '', 'g') AS DOUBLE) BETWEEN 500 AND 30000;
""")

# Load skills mapping
con.execute(f"CREATE OR REPLACE VIEW job_skills AS SELECT * FROM read_csv_auto('{skills_csv}')")
print("Data enriched and seniority classification complete.")


Data enriched and seniority classification complete.


## 4. Industry Filtering Logic
The dashboard needs to extract unique industry categories. Since the data is stored in a structured string format (e.g., category:IT), we use regexp_extract_all and unnest.

In [4]:
# Extract unique categories for the selection menu
query_cats = """
WITH cleaned_data AS (
    SELECT categories FROM sg_jobs WHERE categories IS NOT NULL AND categories != ''
)
SELECT DISTINCT trim(unnest(regexp_extract_all(categories, 'category:([^},]+)', 1))) as cat
FROM cleaned_data ORDER BY cat
"""
categories = con.execute(query_cats).df()['cat'].tolist()
print(f"Found {len(categories)} unique industries.")

Found 43 unique industries.


## 5. The Analysis Engine: Skill & Salary Join
This is the core of the application. We join the job postings with the skills list by matching job titles. We then calculate the Employability Score.

In [5]:
# Example: Analyzing 'Information Technology'
selected_categories = ['Information Technology']
cats_list = str(selected_categories)

query_analysis = f"""
SELECT 
    trim(unnest(string_split(js.job_skills, ','))) as skill,
    s.average_salary,
    s.seniority
FROM sg_jobs s
INNER JOIN job_skills js ON lower(trim(s.title)) = lower(replace(js.job_keyword, '-', ' '))
WHERE list_intersect(regexp_extract_all(s.categories, 'category:([^}},]+)', 1), {cats_list}::VARCHAR[]) != []
AND s.average_salary IS NOT NULL
"""

raw_df = con.execute(query_analysis).df()

# Calculate Score
stats = raw_df.groupby('skill').agg(
    postings=('skill', 'count'),
    avg_salary=('average_salary', 'mean'),
    entry_count=('seniority', lambda x: (x == 'Entry-level').sum())
).reset_index()

# Normalize and Weight (50% Demand, 30% Salary, 20% Accessibility)
stats = stats[stats['postings'] > 2]
stats['entry_share'] = stats['entry_count'] / stats['postings']

for col in ['postings', 'avg_salary', 'entry_share']:
    stats[f'{col}_norm'] = (stats[col] - stats[col].min()) / (stats[col].max() - stats[col].min())

stats['employability_score'] = (stats['postings_norm'] * 0.5 + stats['avg_salary_norm'] * 0.3 + stats['entry_share_norm'] * 0.2)
stats = stats.sort_values('employability_score', ascending=False)
stats.head(5)

,skill,postings,avg_salary,entry_count,entry_share,postings_norm,avg_salary_norm,entry_share_norm,employability_score
4702,Teamwork,1250,7709.332800,18,0.014400,1.000000,0.567473,0.021600,0.674562
3690,Project Management,1046,8577.552581,4,0.003824,0.836276,0.654995,0.005736,0.615784
2580,Java,1026,7632.796296,12,0.011696,0.820225,0.559758,0.017544,0.581548
2672,Leadership,956,8487.426778,4,0.004184,0.764045,0.645910,0.006276,0.577051
1015,Communication,926,8195.078834,8,0.008639,0.739968,0.616439,0.012959,0.557508


## 6. Visualizing Market Insights
A. High-Value Skill Ranking
This visualization helps users identify which skills provide the best "bang for their buck."

In [6]:
top_10 = stats.head(10)
fig_score = px.bar(
    top_10, x='employability_score', y='skill', orientation='h',
    color='avg_salary', color_continuous_scale='RdYlGn',
    title='Top Skills by Employability Score'
)
fig_score.show()

### B. Career Pathways
A dual-axis chart showing how the volume of job opportunities decreases as seniority (and salary) increases.

In [7]:
pathway_df = raw_df.groupby('seniority').agg(
    postings=('seniority', 'count'),
    median_salary=('average_salary', 'median')
).reindex(['Entry-level', 'Mid-level', 'Senior-level']).reset_index()

fig_pathway = go.Figure()
fig_pathway.add_trace(go.Bar(x=pathway_df['seniority'], y=pathway_df['postings'], name='Job Volume', marker_color='#2a9d8f'))
fig_pathway.add_trace(go.Scatter(x=pathway_df['seniority'], y=pathway_df['median_salary'], name='Median Salary', yaxis='y2', line=dict(color='#e45756', width=4)))

fig_pathway.update_layout(
    title='Career Progression: Volume vs Reward',
    yaxis=dict(title="Volume"),
    yaxis2=dict(title="Salary (S$)", overlaying='y', side='right')
)
fig_pathway.show()